# 01 — Construção da Base de Conhecimento (Olist Reviews)

Este notebook documenta e executa a etapa de **Construção da Base de Conhecimento** para a solução de *Voice of Customer Intelligence com RAG* sobre o dataset público da Olist (`olist_order_reviews_dataset.csv`).

### Etapas contempladas:
1. **Seleção dos dados:** inspeção das colunas originais e recorte dos atributos textuais e de rastreabilidade.
2. **Tratamento de valores ausentes:** consolidação entre título (`review_comment_title`) e mensagem (`review_comment_message`) e descarte de avaliações sem conteúdo textual.
3. **Limpeza e preparação dos textos:** normalização Unicode NFC, remoção de caracteres de controle, filtragem de comentários monossilábicos/não informativos e desduplicação.
4. **Organização dos documentos:** estruturação em objetos `Document` do LangChain com metadados (`review_id`, `order_id`, `review_score`, `review_creation_date`).
5. **Criação dos embeddings e indexação vetorial:** vetorização com `OpenAIEmbeddings` (`text-embedding-3-small`) e persistência local no `FAISS`.

## 1. Carregamento e Auditoria da Base Bruta

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DEFAULT_RAW_CSV, DEFAULT_PROCESSED_CSV, CLEANING_AUDIT_PATH
from src.preprocessing.loader import load_and_clean_reviews, build_documents_from_df
import pandas as pd

df_raw = pd.read_csv(DEFAULT_RAW_CSV)
print(f"Total de registros brutos: {len(df_raw):,}")
print(f"Colunas disponíveis: {list(df_raw.columns)}")

Total de registros brutos: 100,000
Colunas disponíveis: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']


## 2. Limpeza Textual, Tratamento de Nulos e Desduplicação

In [2]:
df_clean, audit = load_and_clean_reviews(
    raw_csv_path=DEFAULT_RAW_CSV,
    save_processed_path=DEFAULT_PROCESSED_CSV,
    audit_path=CLEANING_AUDIT_PATH,
)

print("=== Relatório de Auditoria da Preparação da Base ===")
print(f"Total bruto:                      {audit['total_raw_rows']:,}")
print(f"Sem título (nulos):               {audit['null_title_rows']:,}")
print(f"Sem mensagem (nulos):             {audit['null_message_rows']:,}")
print(f"Título e mensagem ambos nulos:    {audit['both_title_and_message_null_rows']:,}")
print(f"Com texto não vazio:              {audit['non_empty_text_rows']:,}")
print(f"Removidos (curtos/sem semântica): {audit['removed_uninformative_or_short_rows']:,}")
print(f"Removidos (textos duplicados):    {audit['removed_duplicate_texts']:,}")
print(f"Avaliações únicas indexáveis:     {audit['final_indexed_reviews']:,}")

=== Relatório de Auditoria da Preparação da Base ===
Total bruto:                      100,000
Sem título (nulos):               88,285
Sem mensagem (nulos):             58,247
Título e mensagem ambos nulos:    56,518
Com texto não vazio:              43,463
Removidos (curtos/sem semântica): 3,446
Removidos (textos duplicados):    3,441
Avaliações únicas indexáveis:     36,576


## 3. Distribuição das Avaliações Limpas por Nota (`review_score`)

In [3]:
total_clean = len(df_clean)
for score, count in sorted(audit["score_distribution"].items(), key=lambda x: int(x[0])):
    pct = (count / total_clean) * 100
    print(f"Nota {score}: {count:6,d} avaliações ({pct:5.2f}%)")

Nota 1:  8,739 avaliações (23.89%)
Nota 2:  2,118 avaliações ( 5.79%)
Nota 3:  3,336 avaliações ( 9.12%)
Nota 4:  5,092 avaliações (13.92%)
Nota 5: 17,291 avaliações (47.27%)


## 4. Organização em Documentos LangChain e Indexação Vetorial (FAISS)

In [4]:
documents = build_documents_from_df(df_clean)
print(f"Total de objetos Document criados: {len(documents):,}")
print("Exemplo de Document[0]:")
print(f"  page_content: {documents[0].page_content!r}")
print(f"  metadata:     {documents[0].metadata}")

Total de objetos Document criados: 36,576
Exemplo de Document[0]:
  page_content: 'Recebi bem antes do prazo estipulado.'
  metadata:     {'review_id': 'e64fb393e7b32834bb789ff8bb30750e', 'order_id': '658677c97b385a9be170737859d3511b', 'review_score': 5, 'review_creation_date': '2017-04-21'}


In [5]:
import os
from src.config import DEFAULT_FAISS_INDEX_DIR
from src.indexing.embeddings import get_embeddings_model
from src.indexing.vector_store import build_faiss_index

if os.getenv("OPENAI_API_KEY") and os.getenv("OPENAI_API_KEY") != "sua-chave-openai-aqui":
    embeddings = get_embeddings_model()
    vector_store = build_faiss_index(documents, embeddings, save_dir=DEFAULT_FAISS_INDEX_DIR)
    print(f"Índice FAISS persistido com sucesso em: {DEFAULT_FAISS_INDEX_DIR}")
else:
    print("Para indexar no FAISS com OpenAIEmbeddings, certifique-se de configurar OPENAI_API_KEY no .env e execute:")
    print("python -m src.cli ingest")

Para indexar no FAISS com OpenAIEmbeddings, certifique-se de configurar OPENAI_API_KEY no .env e execute:
python -m src.cli ingest


### Q&A
- **Como foram tratados os valores ausentes e ruídos nas avaliações da Olist?** As colunas `review_comment_title` e `review_comment_message` foram consolidadas de forma complementar; os 56.518 registros sem nenhum texto foram descartados, assim como 3.446 avaliações curtas sem valor semântico (menos de 8 caracteres ou menos de 2 palavras alfanuméricas) e 3.441 duplicatas exatas.
- **Como os documentos foram organizados para o RAG?** Cada avaliação única foi encapsulada em um objeto `Document` do LangChain contendo o relato higienizado em `page_content` e os metadados de rastreabilidade (`review_id`, `order_id`, `review_score` e `review_creation_date`).

### Data Analysis Key Findings
- Do total de 100.000 avaliações brutas da Olist, 43.463 possuem conteúdo textual preenchido e 36.576 constituem relatos únicos e informativos aptos para indexação semântica.
- A distribuição das avaliações textuais informativas apresenta polarização clara: 47,27% são avaliações nota 5 (17.291 relatos) e 23,89% são avaliações nota 1 (8.739 relatos), concentrando tanto elogios detalhados quanto reclamações críticas de entrega e qualidade.

### Insights or Next Steps
- A remoção das 3.441 mensagens duplicadas genéricas evita que o buscador vetorial sature o *top-k* com textos repetitivos, aumentando a diversidade de evidências enviadas ao LLM.
- O conjunto de 36.576 documentos estruturados alimenta diretamente o pipeline RAG demonstrado no notebook `02_pipeline_rag_exploracao_voc.ipynb`.